In [ ]:
"""
==============================================================
  이탈 예측 최종 통합 모델 (churn_model_final.py)
  실험 결과 기반 베스트 피처 + 베스트 파라미터 집약본
  Target: ROC-AUC 최대화
==============================================================
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')


# ──────────────────────────────────────────────
# 0. 데이터 로드 (경로 수정)
# ──────────────────────────────────────────────
BASE = '/Users/rim/Desktop/workspace/project_1'

train_cust   = pd.read_csv(f'{BASE}/train/train_customer_info.csv')
train_trans  = pd.read_csv(f'{BASE}/train/train_transaction_history.csv')
train_fin    = pd.read_csv(f'{BASE}/train/train_finance_profile.csv')
train_target = pd.read_csv(f'{BASE}/train/train_targets.csv')

test_cust    = pd.read_csv(f'{BASE}/test/test_customer_info.csv')
test_trans   = pd.read_csv(f'{BASE}/test/test_transaction_history.csv')
test_fin     = pd.read_csv(f'{BASE}/test/test_finance_profile.csv')

print("✅ 데이터 로드 완료")


# ──────────────────────────────────────────────
# 1. 피처 엔지니어링 함수
# ──────────────────────────────────────────────
def build_features(cust_df, trans_df, fin_df, target_df=None):
    """
    고객 기본 정보 + 금융 프로필 + 거래 이력을 결합해
    모델 입력용 피처 데이터프레임을 반환합니다.
    target_df가 None이면 테스트 모드로 동작합니다.
    """

    # ── 1-1. 기본 병합 (1:1) ──────────────────
    df = pd.merge(cust_df, fin_df, on='customer_id', how='left')
    if target_df is not None:
        df = pd.merge(df, target_df, on='customer_id', how='left')

    # ── 1-2. 날짜 처리 ────────────────────────
    df['join_date']   = pd.to_datetime(df['join_date'])
    trans_df = trans_df.copy()
    trans_df['trans_date'] = pd.to_datetime(trans_df['trans_date'])
    trans_df['month'] = trans_df['trans_date'].dt.month

    ref_date = trans_df['trans_date'].max()
    df['days_since_joined'] = (ref_date - df['join_date']).dt.days

    # ── 1-3. 범주형 인코딩 ────────────────────
    cat_cols = ['gender', 'region_code', 'prefer_category', 'income_group']
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # ── 1-4. 기본 파생 변수 ───────────────────
    df['net_asset']          = df['total_deposit_balance'] - df['total_loan_balance']
    df['debt_to_deposit']    = df['total_loan_balance'] / (df['total_deposit_balance'] + 1)
    df['cash_service_ratio'] = df['card_cash_service_amt'] / (df['total_deposit_balance'] + 1)

    # 재정 위기 복합 지표 (표준화 합산)
    distress_vars = ['fin_overdue_days', 'card_cash_service_amt',
                     'card_loan_amt', 'total_loan_balance']
    distress_z = df[distress_vars].copy()
    for v in distress_vars:
        mu, sd = distress_z[v].mean(), distress_z[v].std() + 1e-9
        distress_z[v] = (distress_z[v] - mu) / sd
    df['fin_distress_v2'] = distress_z.sum(axis=1)

    # 고위험 자산 플래그
    df['is_high_risk_asset'] = (
        (df['fin_asset_trend_score'] <= -2) & (df['credit_score'] < 600)
    ).astype(int)

    # ── 1-5. 거래 이력 집계 ───────────────────
    # 전체 집계
    agg = trans_df.groupby('customer_id').agg(
        trans_count      = ('trans_id',       'count'),
        amt_sum          = ('trans_amount',    'sum'),
        amt_mean         = ('trans_amount',    'mean'),
        amt_max          = ('trans_amount',    'max'),
        amt_std          = ('trans_amount',    'std'),
        installment_ratio= ('is_installment',  'mean'),
        online_ratio     = ('biz_type',        lambda x: (x == 'Online').mean()),
        recency          = ('trans_date',
                            lambda x: (ref_date - x.max()).days),
    ).reset_index()
    agg['spending_per_trans'] = agg['amt_sum'] / (agg['trans_count'] + 1)
    agg['amt_std'] = agg['amt_std'].fillna(0)

    df = pd.merge(df, agg, on='customer_id', how='left')
    fill_cols = ['trans_count','amt_sum','amt_mean','amt_max',
                 'amt_std','installment_ratio','online_ratio',
                 'recency','spending_per_trans']
    df[fill_cols] = df[fill_cols].fillna(0)

    # ── 1-6. 월별 Wide 피처 (핵심 시계열) ────
    monthly_amt = trans_df.groupby(['customer_id','month'])['trans_amount'] \
        .sum().unstack(fill_value=0)
    monthly_amt.columns = [f'amt_m{int(c)}' for c in monthly_amt.columns]

    monthly_cnt = trans_df.groupby(['customer_id','month'])['trans_id'] \
        .count().unstack(fill_value=0)
    monthly_cnt.columns = [f'cnt_m{int(c)}' for c in monthly_cnt.columns]

    df = df.merge(monthly_amt.reset_index(), on='customer_id', how='left')
    df = df.merge(monthly_cnt.reset_index(), on='customer_id', how='left')

    # 월별 컬럼 결측치 0 처리
    month_cols = [c for c in df.columns if c.startswith('amt_m') or c.startswith('cnt_m')]
    df[month_cols] = df[month_cols].fillna(0)

    # 후반(10~12월) vs 전반(7~9월) 소비 감소율
    first_half_cols  = [c for c in df.columns if c in ['amt_m7','amt_m8','amt_m9']]
    second_half_cols = [c for c in df.columns if c in ['amt_m10','amt_m11','amt_m12']]
    if first_half_cols and second_half_cols:
        df['amt_first_half']    = df[first_half_cols].sum(axis=1)
        df['amt_second_half']   = df[second_half_cols].sum(axis=1)
        df['half_decline_ratio'] = df['amt_second_half'] / (df['amt_first_half'] + 1)

    # 12월 거래 없는 고객 (강력한 이탈 선행 신호)
    last_month_buyers = trans_df[trans_df['month'] == trans_df['month'].max()]['customer_id'].unique()
    df['no_last_month_trans'] = (~df['customer_id'].isin(last_month_buyers)).astype(int)

    # 최근 소비 감소 플래그
    if 'amt_m12' in df.columns and 'amt_m11' in df.columns:
        df['recent_spending_drop'] = (df['amt_m12'] < df['amt_m11']).astype(int)

    # spending trend ratio (최근 1달 vs 이전 5달 평균)
    max_month = trans_df['month'].max()
    recent_sum = trans_df[trans_df['month'] == max_month] \
        .groupby('customer_id')['trans_amount'].sum()
    older_mean = trans_df[trans_df['month'] < max_month] \
        .groupby('customer_id')['trans_amount'].mean()
    df['spending_trend_ratio'] = (
        df['customer_id'].map(recent_sum).fillna(0) /
        (df['customer_id'].map(older_mean).fillna(1) + 1)
    )

    # ── 1-7. 엔트로피 피처 ───────────────────
    def calc_entropy(series):
        counts = series.value_counts(normalize=True)
        return scipy_entropy(counts)

    cat_ent = trans_df.groupby('customer_id')['item_category'].agg(calc_entropy)
    df['category_entropy'] = df['customer_id'].map(cat_ent).fillna(0)

    # ── 1-8. 피어 그룹 내 상대 순위 ──────────
    df['credit_rank_in_income']  = df.groupby('income_group')['credit_score'] \
                                     .rank(pct=True)
    df['deposit_rank_in_region'] = df.groupby('region_code')['total_deposit_balance'] \
                                     .rank(pct=True)
    df['age_group'] = pd.cut(df['age'], bins=[0,30,40,50,100],
                              labels=[0,1,2,3]).astype(float)
    df['loan_rank_in_age']       = df.groupby('age_group')['card_loan_amt'] \
                                     .rank(pct=True)

    # ── 1-9. 타겟 인코딩 (train only, OOF 방식) ─
    if target_df is not None:
        def target_encode_oof(df_, col, target_col='target_churn', n_splits=5):
            encoded = pd.Series(index=df_.index, dtype=float)
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            for tr_idx, val_idx in kf.split(df_):
                mean_map = df_.iloc[tr_idx].groupby(col)[target_col].mean()
                encoded.iloc[val_idx] = df_.iloc[val_idx][col].map(mean_map)
            return encoded.fillna(df_[target_col].mean())

        for col in ['region_code', 'prefer_category', 'income_group']:
            if col in df.columns:
                df[f'{col}_churn_rate'] = target_encode_oof(df, col)

    return df


# ──────────────────────────────────────────────
# 2. 피처 목록 정의
# ──────────────────────────────────────────────
FEATURES = [
    # 금융 핵심
    'total_deposit_balance', 'card_loan_amt', 'credit_score',
    'fin_distress_v2', 'net_asset', 'fin_asset_trend_score',
    'num_active_cards', 'fin_overdue_days',
    'debt_to_deposit', 'cash_service_ratio',

    # 거래 집계
    'trans_count', 'amt_sum', 'amt_mean', 'amt_std',
    'spending_per_trans', 'installment_ratio', 'online_ratio', 'recency',

    # 시계열 (월별 wide)
    'amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12',
    'cnt_m7','cnt_m8','cnt_m9','cnt_m10','cnt_m11','cnt_m12',
    'half_decline_ratio', 'no_last_month_trans',
    'recent_spending_drop', 'spending_trend_ratio',

    # 피어 그룹 순위
    'credit_rank_in_income', 'deposit_rank_in_region', 'loan_rank_in_age',

    # 엔트로피
    'category_entropy',

    # 고객 속성
    'age', 'is_married', 'days_since_joined', 'is_high_risk_asset',
]


# ──────────────────────────────────────────────
# 3. 학습 데이터 구축
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (train)...")
train_df = build_features(train_cust, train_trans, train_fin, train_target)

# 타겟 인코딩 피처 추가
te_features = [f'{c}_churn_rate' for c in ['region_code','prefer_category','income_group']
               if f'{c}_churn_rate' in train_df.columns]

final_features = [f for f in FEATURES + te_features if f in train_df.columns]

X = train_df[final_features].copy()
y = train_df['target_churn']

print(f"✅ 학습 피처 수: {len(final_features)}개 | 샘플 수: {len(X):,}명")
print(f"   이탈율: {y.mean():.3%}")


# ──────────────────────────────────────────────
# 4. 모델 학습 (Optuna Trial 31 베스트 파라미터)
# ──────────────────────────────────────────────
BEST_PARAMS = {
    'objective':       'binary',
    'metric':          'auc',
    'verbosity':       -1,
    'boosting_type':   'gbdt',
    'random_state':    42,
    'n_estimators':    5000,
    # Optuna 최적값
    'learning_rate':   0.014369,
    'num_leaves':      31,
    'min_child_samples': 22,
    'feature_fraction':  0.7661,
    'bagging_fraction':  0.7553,
    'bagging_freq':      6,
    'reg_alpha':         0.0172,
    'reg_lambda':        0.0525,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models    = []
fold_aucs = []

print("\n🚀 5-Fold 교차 검증 시작...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**BEST_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False),
                   lgb.log_evaluation(0)]
    )
    models.append(model)

    pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = pred
    auc = roc_auc_score(y_val, pred)
    fold_aucs.append(auc)
    print(f"  Fold {fold+1}: AUC = {auc:.4f}")

cv_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"  CV AUC (OOF): {cv_auc:.4f}")
print(f"  Fold 평균:    {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"{'='*45}")

# 피처 중요도 Top 15
importances = pd.Series(
    np.mean([m.feature_importances_ for m in models], axis=0),
    index=final_features
).sort_values(ascending=False)
print("\n🔥 피처 중요도 Top 15:")
print(importances.head(15).to_string())


# ──────────────────────────────────────────────
# 5. 테스트 예측 및 제출 파일 생성
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (test)...")
test_df = build_features(test_cust, test_trans, test_fin, target_df=None)

# 타겟 인코딩 — train 전체 통계로 매핑 (test는 OOF 불필요)
for col in ['region_code', 'prefer_category', 'income_group']:
    col_te = f'{col}_churn_rate'
    if col_te in final_features:
        train_mean_map = train_df.groupby(col)['target_churn'].mean()
        test_df[col_te] = test_df[col].map(train_mean_map).fillna(y.mean())

# 없는 컬럼 0으로 패딩
for f in final_features:
    if f not in test_df.columns:
        test_df[f] = 0

X_test = test_df[final_features].copy()

# 5개 fold 모델 평균으로 최종 예측
test_preds = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)

# 제출 파일 저장
submission = pd.DataFrame({
    'customer_id':  test_df['customer_id'],
    'target_churn': test_preds,
    'target_ltv':   0.0   # LTV는 별도 모델 필요 — 일단 0으로 플레이스홀더
})
#submission.to_csv(f'{BASE}/submission_churn.csv', index=False, encoding='utf-8-sig')

#print(f"\n✅ 제출 파일 저장 완료: {BASE}/submission_churn.csv")
#print(f"   예측값 범위: {test_preds.min():.4f} ~ {test_preds.max():.4f}")
#print(f"   예측 이탈율: {test_preds.mean():.3%}")

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

BASE = '/Users/rim/Desktop/workspace/project_1'
train = pd.read_csv(f'{BASE}/train_p.csv')
test  = pd.read_csv(f'{BASE}/test_p.csv')


def competition_score(auc, rmse):
    ltv_part = 1 / (1 + np.log(rmse))
    return 0.5 * auc + 0.5 * ltv_part, ltv_part


# ──────────────────────────────────────────────
# 추가 파생 피처 생성 (train/test 동시)
# ──────────────────────────────────────────────
for df_ in [train, test]:
    # 피어 그룹 순위 (0.7947 시절 핵심 피처)
    df_['deposit_rank_in_region'] = df_.groupby('region_code')['total_deposit_balance'] \
                                       .rank(pct=True)
    df_['loan_rank_in_age'] = df_.groupby(
        pd.cut(df_['age'], bins=[0,30,40,50,100], labels=[0,1,2,3])
    )['card_loan_amt'].rank(pct=True)

    # 순자산 / 소비 관련
    if 'net_asset' not in df_.columns:
        df_['net_asset'] = df_['total_deposit_balance'] - df_['total_loan_balance']
    df_['spending_cv']       = df_['amt_std'] / (df_['amt_mean'] + 1)
    df_['monthly_avg_spend'] = df_['amt_sum']  / (df_['active_months'] + 1)
    df_['spend_to_deposit']  = df_['amt_sum']  / (df_['total_deposit_balance'] + 1)
    df_['loyalty_index']     = df_['trans_count'] / (df_['days_since_joined'] / 30 + 1)
    df_['dec_growth']        = df_['amt_m12'] / (df_['amt_m11'] + 1)
    df_['no_last_trans']     = (df_['cnt_m12'] == 0).astype(int)
    df_['recent_drop']       = (df_['amt_m12'] < df_['amt_m11']).astype(int)
    df_['high_value_ratio']  = df_['amt_max'] / (df_['amt_mean'] + 1)

    # [요청 추가 1] 연속 감소 달 수
    months = ['amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12']
    df_['consecutive_decline'] = sum(
        (df_[months[i]] < df_[months[i-1]]).astype(int)
        for i in range(1, len(months))
    )
    
    # [요청 추가 2] 최근 3달 vs 이전 3달 비율
    df_['recent3_sum']  = df_[['amt_m10','amt_m11','amt_m12']].sum(axis=1)
    df_['early3_sum']   = df_[['amt_m7','amt_m8','amt_m9']].sum(axis=1)
    df_['trend_ratio']  = df_['recent3_sum'] / (df_['early3_sum'] + 1)
    df_['recent_zero_months'] = (
        (df_['amt_m11'] == 0).astype(int) + 
        (df_['amt_m12'] == 0).astype(int)
    )
    
    # [요청 추가 3] 교차 피처
    df_['credit_x_trend']      = df_['credit_score'] * df_['fin_asset_trend_score']
    df_['distress_x_recency']  = df_['fin_distress_v2'] * np.log1p(df_['recency'])
    df_['tenure_x_drop']       = df_['days_since_joined'] * df_['recent_drop']
    df_['deposit_z_in_income'] = df_.groupby('income_group')['total_deposit_balance'] \
        .transform(lambda x: (x - x.mean()) / (x.std() + 1))

# ──────────────────────────────────────────────
# 1. CHURN 모델
# ──────────────────────────────────────────────
print("=" * 55)
print("  🎯 Task 1: Churn 이탈 예측")
print("=" * 55)

# 0.7947 나왔을 때 피처 + 추가 피처
CH_FEATS = [
    # 금융 위기 (핵심)
    'fin_distress_v2', 'fin_overdue_days', 'card_loan_amt',
    'fin_asset_trend_score', 'debt_to_deposit', 'cash_service_ratio',
    # 자산 방어력
    'total_deposit_balance', 'net_asset', 'credit_score', 'num_active_cards',
    # 거래 감소 신호
    'recency', 'no_last_trans', 'recent_drop',
    'trans_count', 'spending_per_trans',
    'amt_m10', 'amt_m11', 'amt_m12',
    'cnt_m10', 'cnt_m11', 'cnt_m12',
    # 피어 그룹 (핵심)
    'credit_rank_in_income', 'deposit_rank_in_region', 'loan_rank_in_age',
    # 고객 속성
    'age', 'is_married', 'days_since_joined',
    'gender', 'region_code', 'prefer_category', 'income_group',
]

# [요청 추가] CH_FEATS에 신규 피처 추가
new_feats = [
    'consecutive_decline', 'trend_ratio', 'recent_zero_months',
    'credit_x_trend', 'distress_x_recency', 'tenure_x_drop',
    'deposit_z_in_income'
]
CH_FEATS = CH_FEATS + [f for f in new_feats if f in train.columns]

# 타겟 인코딩 (OOF)
for c in ['region_code', 'prefer_category', 'income_group']:
    f = f'{c}_churn_rate'
    train[f] = 0.0
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for tr, val in kf.split(train):
        mapping = train.iloc[tr].groupby(c)['target_churn'].mean()
        train.loc[train.index[val], f] = train.loc[train.index[val], c].map(mapping)
    train[f].fillna(train['target_churn'].mean(), inplace=True)
    test[f] = test[c].map(train.groupby(c)['target_churn'].mean()) \
                     .fillna(train['target_churn'].mean())
    CH_FEATS.append(f)

# 없는 컬럼 제외
CH_FEATS = [f for f in CH_FEATS if f in train.columns]
for f in CH_FEATS:
    if f not in test.columns:
        test[f] = 0

print(f"  Churn 피처 수: {len(CH_FEATS)}개")

# 5-Fold OOF 학습
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_churn  = np.zeros(len(train))
test_churn = np.zeros(len(test))
fold_aucs  = []

CHURN_PARAMS = {
    'n_estimators':    5000,
    'learning_rate':   0.014369,
    'num_leaves':      31,
    'min_child_samples': 22,
    'feature_fraction':  0.7661,
    'bagging_fraction':  0.7553,
    'bagging_freq':      6,
    'reg_alpha':         0.0172,
    'reg_lambda':        0.0525,
    'random_state':      42,
    'verbosity':        -1,
}

for fold, (tr, val) in enumerate(skf.split(train[CH_FEATS], train['target_churn'])):
    m = lgb.LGBMClassifier(**CHURN_PARAMS)
    m.fit(
        train[CH_FEATS].iloc[tr], train['target_churn'].iloc[tr],
        eval_set=[(train[CH_FEATS].iloc[val], train['target_churn'].iloc[val])],
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
    )
    oof_churn[val]  = m.predict_proba(train[CH_FEATS].iloc[val])[:, 1]
    test_churn     += m.predict_proba(test[CH_FEATS])[:, 1] / 5
    auc = roc_auc_score(train['target_churn'].iloc[val], oof_churn[val])
    fold_aucs.append(auc)
    print(f"  Fold {fold+1}: AUC = {auc:.4f}")

cv_auc = roc_auc_score(train['target_churn'], oof_churn)
print(f"\n  ✅ Churn CV AUC: {cv_auc:.4f}  (평균: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f})")

# 피처 중요도 (상위 10)
imp_ch = pd.Series(m.feature_importances_, index=CH_FEATS).sort_values(ascending=False)
print(f"\n  🔥 Churn 피처 중요도 Top 10:")
for i, (feat, val_) in enumerate(imp_ch.head(10).items()):
    print(f"    {i+1:2}. {feat:<35} {val_:.1f}")


# ──────────────────────────────────────────────
# 2. LTV 모델
# ──────────────────────────────────────────────
print("\n" + "=" * 55)
print("  💰 Task 2: LTV 생애 가치 예측")
print("=" * 55)

y_ltv = train['target_ltv']
print(f"\n  LTV 분포: 평균 {y_ltv.mean():,.0f}원 | 왜도 {y_ltv.skew():.2f}")
print(f"  log 변환 후 왜도: {np.log1p(y_ltv).skew():.2f}")

LTV_FEATS = [
    # 소비 핵심
    'amt_sum', 'amt_mean', 'amt_max', 'amt_std',
    'spending_per_trans', 'spending_cv',
    # 월별 전체
    'amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12',
    'cnt_m7','cnt_m8','cnt_m9','cnt_m10','cnt_m11','cnt_m12',
    # 트렌드 요약
    'amt_first_half','amt_second_half','half_growth_ratio',
    'active_months', 'dec_growth',
    # 소비 여력
    'total_deposit_balance','net_asset','credit_score',
    'num_active_cards','fin_asset_trend_score',
    # LTV 특화 파생
    'spend_to_deposit','monthly_avg_spend',
    'loyalty_index','high_value_ratio',
    # 상대 순위
    'deposit_rank_in_region','spend_rank_in_region','credit_rank_in_income',
    # 고객 속성
    'age','is_married','days_since_joined',
    'gender','region_code','prefer_category','income_group',
]

LTV_FEATS = [f for f in LTV_FEATS if f in train.columns]
for f in LTV_FEATS:
    if f not in test.columns:
        test[f] = 0

print(f"  LTV 피처 수: {len(LTV_FEATS)}개")

y_log = np.log1p(y_ltv)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_ltv_log = np.zeros(len(train))
test_ltv    = np.zeros(len(test))
fold_rmses  = []

LTV_PARAMS = {
    'objective':       'regression',
    'n_estimators':    5000,
    'learning_rate':   0.01,
    'num_leaves':      63,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'random_state':      42,
    'verbosity':        -1,
}

for fold, (tr, val) in enumerate(kf.split(train[LTV_FEATS])):
    m = lgb.LGBMRegressor(**LTV_PARAMS)
    m.fit(
        train[LTV_FEATS].iloc[tr], y_log.iloc[tr],
        eval_set=[(train[LTV_FEATS].iloc[val], y_log.iloc[val])],
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
    )
    oof_ltv_log[val] = m.predict(train[LTV_FEATS].iloc[val])
    test_ltv        += np.expm1(m.predict(test[LTV_FEATS])) / 5

    fold_rmse = np.sqrt(mean_squared_error(
        y_ltv.iloc[val], np.expm1(oof_ltv_log[val])
    ))
    fold_rmses.append(fold_rmse)
    print(f"  Fold {fold+1}: RMSE = {fold_rmse:>12,.0f}원")

oof_ltv_final = np.maximum(np.expm1(oof_ltv_log), 0)
test_ltv      = np.maximum(test_ltv, 0)
cv_rmse       = np.sqrt(mean_squared_error(y_ltv, oof_ltv_final))

print(f"\n  ✅ LTV CV RMSE: {cv_rmse:,.0f}원")

# 피처 중요도
imp_ltv = pd.Series(m.feature_importances_, index=LTV_FEATS).sort_values(ascending=False)
print(f"\n  🔥 LTV 피처 중요도 Top 10:")
for i, (feat, val_) in enumerate(imp_ltv.head(10).items()):
    print(f"    {i+1:2}. {feat:<35} {val_:.1f}")


# ──────────────────────────────────────────────
# 3. 대회 점수 계산
# ──────────────────────────────────────────────
total, ltv_part = competition_score(cv_auc, cv_rmse)

print("\n" + "=" * 55)
print("  📊 대회 점수 계산 (CV 기준)")
print("=" * 55)
print(f"  Churn AUC:    {cv_auc:.4f}")
print(f"  LTV RMSE:     {cv_rmse:>12,.0f}원")
print(f"  LTV 부분점수: {ltv_part:.6f}")
print(f"  {'─'*40}")
print(f"  통합 Score:   {total:.4f}")
print(f"  = 0.5 × {cv_auc:.4f} + 0.5 × {ltv_part:.4f}")
print(f"  = {0.5*cv_auc:.4f} + {0.5*ltv_part:.4f}")


# ──────────────────────────────────────────────
# 4. 제출 파일 생성
# ──────────────────────────────────────────────
# submission = pd.DataFrame({
#    'customer_id':  test['customer_id'],
#    'target_churn': np.clip(test_churn, 0, 1),
#    'target_ltv':   np.clip(test_ltv, 0, None),
# })

# out_path = f'{BASE}/submission_final.csv'
# submission.to_csv(out_path, index=False, encoding='utf-8-sig')

print("\n" + "=" * 55)
# print(f"  ✅ 제출 파일: {out_path}")
print(f"  ✅ 총 {len(submission):,}명")
print(f"  Churn 범위: {test_churn.min():.4f} ~ {test_churn.max():.4f}")
print(f"  Churn 평균: {test_churn.mean():.4f}  (train: {train['target_churn'].mean():.4f})")
print(f"  LTV 범위:   {test_ltv.min():,.0f} ~ {test_ltv.max():,.0f}원")
print(f"  LTV 평균:   {test_ltv.mean():,.0f}원  (train: {y_ltv.mean():,.0f}원)")
print("=" * 55)

  🎯 Task 1: Churn 이탈 예측
  Churn 피처 수: 41개
  Fold 1: AUC = 0.7993
  Fold 2: AUC = 0.7842
  Fold 3: AUC = 0.8024
  Fold 4: AUC = 0.8042
  Fold 5: AUC = 0.7961

  ✅ Churn CV AUC: 0.7971  (평균: 0.7973 ± 0.0071)

  🔥 Churn 피처 중요도 Top 10:
     1. total_deposit_balance               624.0
     2. deposit_rank_in_region              593.0
     3. loan_rank_in_age                    537.0
     4. fin_asset_trend_score               533.0
     5. trend_ratio                         528.0
     6. credit_rank_in_income               509.0
     7. card_loan_amt                       467.0
     8. credit_score                        462.0
     9. amt_m10                             442.0
    10. deposit_z_in_income                 437.0

  💰 Task 2: LTV 생애 가치 예측

  LTV 분포: 평균 1,239,290원 | 왜도 2.12
  log 변환 후 왜도: -1.14
  LTV 피처 수: 42개
  Fold 1: RMSE =    1,540,541원
  Fold 2: RMSE =    1,485,646원
  Fold 3: RMSE =    1,469,405원
  Fold 4: RMSE =    1,511,371원
  Fold 5: RMSE =    1,518,695원

  ✅ LTV CV RMS

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

BASE = '/Users/rim/Desktop/workspace/project_1'
train = pd.read_csv(f'{BASE}/train_p.csv')
test  = pd.read_csv(f'{BASE}/test_p.csv')

# ──────────────────────────────────────────────
# 1. 파생 피처 생성 (기존 로직 + 요청 추가분 통합)
# ──────────────────────────────────────────────
for df_ in [train, test]:
    # 기존 핵심 피처
    df_['deposit_rank_in_region'] = df_.groupby('region_code')['total_deposit_balance'].rank(pct=True)
    df_['loan_rank_in_age'] = df_.groupby(pd.cut(df_['age'], bins=[0,30,40,50,100], labels=[0,1,2,3]))['card_loan_amt'].rank(pct=True)
    if 'net_asset' not in df_.columns:
        df_['net_asset'] = df_['total_deposit_balance'] - df_['total_loan_balance']
    
    df_['spending_cv']       = df_['amt_std'] / (df_['amt_mean'] + 1)
    df_['monthly_avg_spend'] = df_['amt_sum'] / (df_['active_months'] + 1)
    df_['spend_to_deposit']  = df_['amt_sum'] / (df_['total_deposit_balance'] + 1)
    df_['loyalty_index']     = df_['trans_count'] / (df_['days_since_joined'] / 30 + 1)
    df_['dec_growth']        = df_['amt_m12'] / (df_['amt_m11'] + 1)
    df_['no_last_trans']     = (df_['cnt_m12'] == 0).astype(int)
    df_['recent_drop']       = (df_['amt_m12'] < df_['amt_m11']).astype(int)
    df_['high_value_ratio']  = df_['amt_max'] / (df_['amt_mean'] + 1)

    # [추가 요청 로직]
    months = ['amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12']
    df_['consecutive_decline'] = sum((df_[months[i]] < df_[months[i-1]]).astype(int) for i in range(1, len(months)))
    df_['recent3_sum']  = df_[['amt_m10','amt_m11','amt_m12']].sum(axis=1)
    df_['early3_sum']   = df_[['amt_m7','amt_m8','amt_m9']].sum(axis=1)
    df_['trend_ratio']  = df_['recent3_sum'] / (df_['early3_sum'] + 1)
    df_['recent_zero_months'] = (df_['amt_m11'] == 0).astype(int) + (df_['amt_m12'] == 0).astype(int)
    df_['credit_x_trend']      = df_['credit_score'] * df_['fin_asset_trend_score']
    df_['distress_x_recency']  = df_['fin_distress_v2'] * np.log1p(df_['recency'])
    df_['tenure_x_drop']       = df_['days_since_joined'] * df_['recent_drop']
    df_['deposit_z_in_income'] = df_.groupby('income_group')['total_deposit_balance'].transform(lambda x: (x - x.mean()) / (x.std() + 1))

# 타겟 인코딩
for c in ['region_code', 'prefer_category', 'income_group']:
    f = f'{c}_churn_rate'
    train[f] = 0.0
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for tr, val in kf.split(train):
        mapping = train.iloc[tr].groupby(c)['target_churn'].mean()
        train.loc[train.index[val], f] = train.loc[train.index[val], c].map(mapping)
    train[f].fillna(train['target_churn'].mean(), inplace=True)
    test[f] = test[c].map(train.groupby(c)['target_churn'].mean()).fillna(train['target_churn'].mean())

# ──────────────────────────────────────────────
# 2. CHURN 피처 영향도 분석 (Leave-One-Out)
# ──────────────────────────────────────────────
CH_FEATS = [
    'fin_distress_v2', 'fin_overdue_days', 'card_loan_amt', 'fin_asset_trend_score', 'debt_to_deposit', 
    'cash_service_ratio', 'total_deposit_balance', 'net_asset', 'credit_score', 'num_active_cards',
    'recency', 'no_last_trans', 'recent_drop', 'trans_count', 'spending_per_trans',
    'amt_m10', 'amt_m11', 'amt_m12', 'cnt_m10', 'cnt_m11', 'cnt_m12',
    'credit_rank_in_income', 'deposit_rank_in_region', 'loan_rank_in_age',
    'age', 'is_married', 'days_since_joined', 'gender', 'region_code', 'prefer_category', 'income_group',
    'region_code_churn_rate', 'prefer_category_churn_rate', 'income_group_churn_rate',
    'consecutive_decline', 'trend_ratio', 'recent_zero_months', 'credit_x_trend', 
    'distress_x_recency', 'tenure_x_drop', 'deposit_z_in_income'
]
CH_FEATS = [f for f in CH_FEATS if f in train.columns]

print("\n" + "=" * 55)
print("  🔍 피처 영향도 분석 (노이즈 제거 진행 중...)")
print("=" * 55)

base_auc = 0.7974
results = {}
for feat in CH_FEATS:
    feats_without = [f for f in CH_FEATS if f != feat]
    fold_aucs = []
    skf_fs = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for tr, val in skf_fs.split(train[feats_without], train['target_churn']):
        m_fs = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, verbosity=-1, random_state=42)
        m_fs.fit(train[feats_without].iloc[tr], train['target_churn'].iloc[tr])
        fold_aucs.append(roc_auc_score(train['target_churn'].iloc[val], m_fs.predict_proba(train[feats_without].iloc[val])[:, 1]))
    results[feat] = np.mean(fold_aucs)

final_ch_feats = []
print(f"\n  기준 AUC: {base_auc:.4f}")
for feat, auc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    diff = auc - base_auc
    if diff > 0:
        print(f"  🗑️ 제거 후보: {feat:<25} {auc:.4f} ({diff:+.4f})")
    else:
        print(f"  ✅ 유지     : {feat:<25} {auc:.4f} ({diff:+.4f})")
        final_ch_feats.append(feat)

# ──────────────────────────────────────────────
# 3. 최종 메인 모델 학습 (최적화된 피처셋 사용)
# ──────────────────────────────────────────────
print("\n" + "=" * 55)
print(f"  🎯 최적 피처 ({len(final_ch_feats)}개)로 최종 학습 시작")
print("=" * 55)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_churn, test_churn = np.zeros(len(train)), np.zeros(len(test))

for fold, (tr, val) in enumerate(skf.split(train[final_ch_feats], train['target_churn'])):
    m = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.014369, num_leaves=31, 
                           min_child_samples=22, feature_fraction=0.7661, bagging_fraction=0.7553, 
                           bagging_freq=6, reg_alpha=0.0172, reg_lambda=0.0525, random_state=42, verbosity=-1)
    m.fit(train[final_ch_feats].iloc[tr], train['target_churn'].iloc[tr],
          eval_set=[(train[final_ch_feats].iloc[val], train['target_churn'].iloc[val])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_churn[val] = m.predict_proba(train[final_ch_feats].iloc[val])[:, 1]
    test_churn += m.predict_proba(test[final_ch_feats])[:, 1] / 5

cv_auc = roc_auc_score(train['target_churn'], oof_churn)
print(f"\n  ✅ Churn CV AUC: {cv_auc:.4f}")

# ──────────────────────────────────────────────
# 4. LTV 모델 (동일 피처셋 유지)
# ──────────────────────────────────────────────
LTV_FEATS = [
    'amt_sum', 'amt_mean', 'amt_max', 'amt_std', 'spending_per_trans', 'spending_cv',
    'amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12',
    'cnt_m7','cnt_m8','cnt_m9','cnt_m10','cnt_m11','cnt_m12',
    'amt_first_half','amt_second_half','half_growth_ratio', 'active_months', 'dec_growth',
    'total_deposit_balance','net_asset','credit_score','num_active_cards','fin_asset_trend_score',
    'spend_to_deposit','monthly_avg_spend','loyalty_index','high_value_ratio',
    'deposit_rank_in_region','spend_rank_in_region','credit_rank_in_income',
    'age','is_married','days_since_joined','gender','region_code','prefer_category','income_group'
]
LTV_FEATS = [f for f in LTV_FEATS if f in train.columns]
y_ltv, y_log = train['target_ltv'], np.log1p(train['target_ltv'])
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_ltv_log, test_ltv = np.zeros(len(train)), np.zeros(len(test))

for fold, (tr, val) in enumerate(kf.split(train[LTV_FEATS])):
    m = lgb.LGBMRegressor(objective='regression', n_estimators=5000, learning_rate=0.01, num_leaves=63,
                          feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5, random_state=42, verbosity=-1)
    m.fit(train[LTV_FEATS].iloc[tr], y_log.iloc[tr], eval_set=[(train[LTV_FEATS].iloc[val], y_log.iloc[val])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_ltv_log[val] = m.predict(train[LTV_FEATS].iloc[val])
    test_ltv += np.expm1(m.predict(test[LTV_FEATS])) / 5

cv_rmse = np.sqrt(mean_squared_error(y_ltv, np.maximum(np.expm1(oof_ltv_log), 0)))
print(f"  ✅ LTV CV RMSE: {cv_rmse:,.0f}원")

# ──────────────────────────────────────────────
# 5. 최종 결과 저장
# ──────────────────────────────────────────────
ltv_part = 1 / (1 + np.log(cv_rmse))
print(f"\n  📊 통합 Score: {0.5 * cv_auc + 0.5 * ltv_part:.4f}")